# Maamar: Phase 2 design-decision labels (rules v1.2)

This notebook generates the Phase 2 labels of the Maamar dataset from the Phase 1 site indicators, following the decision rules document **`maamar_rules_v1.2.pdf`**.

| | |
|---|---|
| **Input** | `villas_all_processed.csv` (Phase 1, 22,939 villa plots) · `riyadh_parcels_all.parquet` (plot boundaries) · Riyadh TMYx EPW weather file · Riyadh district boundaries (National Address, via the open `homaily/Saudi-Arabia-Regions-Cities-and-Districts` dataset) |
| **Output** | `phase2_labels_v1.2.csv` (one row per plot, 58 columns) · `phase2_facade_labels_v1.2.csv` (one row per facade, 13 columns) · `phase2_trace_v1.2.csv` (intermediate values) · `summary_v1.2.json` (class distributions) |

Decisions produced:

1. **Shading device** per facade
2. **Window-to-wall ratio (WWR)** per facade
3. **Garden location** per plot
4. **Main facade orientation** per plot
5. **Materials** per facade: wall, thermal glass, acoustic glass

The labels are deterministic: the same inputs always give the same labels.

In [ ]:
# Colab: install the libraries that are not preinstalled
!pip -q install pvlib geopandas shapely pyarrow

## 1. Settings and input files

Set `DATA_DIR` to the folder that holds the input files. On Colab, mount Google Drive and point it at the shared project folder.

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, geopandas as gpd, pvlib
from sklearn.model_selection import GroupKFold

# from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = os.environ.get('MAAMAR_DATA', '/content/drive/MyDrive/Maamar')   # change if needed
OUT_DIR = os.environ.get('MAAMAR_OUT', 'phase2_v1.2')
os.makedirs(OUT_DIR, exist_ok=True)

PHASE1   = f'{DATA_DIR}/dataset_summary/villas_all_processed.csv'
PARCELS  = f'{DATA_DIR}/riyadh_parcels_all.parquet'
EPW      = f'{DATA_DIR}/SAU_RI_Riyadh-Khalid.Intl.AP.404370_TMYx.2011-2025.epw'
DISTRICTS = os.environ.get('MAAMAR_DISTRICTS',
    'https://raw.githubusercontent.com/homaily/Saudi-Arabia-Regions-Cities-and-Districts/master/geojson/districts.geojson')

D = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']          # the 8 facade directions
NOM = dict(zip(D, range(0, 360, 45)))                     # nominal azimuth of each direction

## 2. Load Phase 1 and build the plot ID

The ID is built from the plot coordinates (6 decimals). The few duplicate coordinates get a suffix `_0`, `_1`, …

In [2]:
df = pd.read_csv(PHASE1, low_memory=False)
df.insert(0, 'ID', df.lat.round(6).astype(str) + '_' + df.lon.round(6).astype(str))
dup = df.ID.duplicated(keep=False)
df.loc[dup, 'ID'] = df.loc[dup, 'ID'] + '_' + df[dup].groupby('ID').cumcount().astype(str)
n = len(df)
print(n, 'plots')

22939 plots


## 3. Facade angles from the plot boundary

Each plot point is matched to the parcel that contains it, or to the nearest parcel within 30 m. The plot axis is the bearing of the first edge of the parcel's **minimum rotated rectangle** (mod 90°). Each facade direction is snapped to the closest of the 4 axis directions; if none is within 22.5°, the nominal direction is kept.

In [3]:
par = gpd.read_parquet(PARCELS)[['geometry']]
pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(par.crs)
j = gpd.sjoin(pts, par, how='left', predicate='within')
j = j[~j.index.duplicated()]
miss = j.index_right.isna()
if miss.any():
    near = gpd.sjoin_nearest(pts[miss.values], par, how='left', max_distance=30)
    near = near[~near.index.duplicated()]
    j.loc[near.index, 'index_right'] = near.index_right

axis = np.full(n, np.nan)
for i, pid in zip(j.index, j.index_right):
    if pd.isna(pid):
        continue
    c = np.array(par.geometry.loc[pid].minimum_rotated_rectangle.exterior.coords)
    e = c[1] - c[0]
    axis[i] = np.degrees(np.arctan2(e[0], e[1])) % 90
df['plot_axis'] = np.round(axis, 1)

def facade_az(ax, d):
    if np.isnan(ax):
        return float(NOM[d])
    cands = [(ax + k * 90) % 360 for k in range(4)]
    diff = [abs((c - NOM[d] + 180) % 360 - 180) for c in cands]
    k = int(np.argmin(diff))
    return round(cands[k], 1) if diff[k] <= 22.5 else float(NOM[d])

for d in D:
    has = df[f'Rad_summer_{d}'].notna()
    df[f'az_{d}'] = np.where(has, [facade_az(a, d) for a in df.plot_axis], np.nan)
print('plot axis from the parcel:', np.isfinite(axis).sum(), 'of', n)

plot axis from the parcel: 22939 of 22939


## 4. Window heat gain and the reference G_REF (rules, p. 1)

Hourly sun from the EPW, **April to October**, daytime hours only. For every facade azimuth (0–359°) and each shading device, the solar gain on a standard window is computed:

- Window 1.5 m high × 1.2 m wide; overhang depth 0.75 m; vertical fins 0.6 m.
- Direct part: DNI × cos(incidence) × unshaded fraction (overhang from the vertical shadow angle, fins from the horizontal angle).
- Diffuse part: DHI × 0.5 × sky factor of the device (none 1, horizontal 0.8, vertical 0.8, egg-crate 0.65).

**G_REF** = the gain of an open north window (azimuth 0°, no device). It is the reference every facade is compared against.

In [4]:
w, meta = pvlib.iotools.read_epw(EPW, coerce_year=2021)
sp = pvlib.solarposition.get_solarposition(w.index, meta['latitude'], meta['longitude'], meta['altitude'])
alt = (90 - sp.apparent_zenith).values
saz = sp.azimuth.values
day = (alt > 0) & w.index.month.isin(range(4, 11))
alt, saz, dni, dhi = alt[day], saz[day], w.dni.values[day], w.dhi.values[day]

WIN_H, WIN_W = 1.5, 1.2
OVER, FIN = 0.75, 0.6
DEV = ['بدون', 'أفقي', 'رأسي', 'شبكي']                 # none, horizontal, vertical, egg-crate
SKY = {'بدون': 1.0, 'أفقي': 0.8, 'رأسي': 0.8, 'شبكي': 0.65}

TAB = np.zeros((360, 4))   # kWh/m2 per facade azimuth and device
for az in range(360):
    gam = (saz - az + 540) % 360 - 180
    cosi = np.clip(np.cos(np.radians(alt)) * np.cos(np.radians(gam)), 0, None)
    with np.errstate(all='ignore'):
        vsa = np.arctan(np.tan(np.radians(alt)) / np.abs(np.cos(np.radians(gam))))
        fv = np.clip(OVER * np.tan(vsa) / WIN_H, 0, 1)
        fh = np.clip(FIN * np.abs(np.tan(np.radians(gam))) / WIN_W, 0, 1)
    fv, fh = np.nan_to_num(fv, nan=1.0), np.nan_to_num(fh, nan=1.0)
    unshaded = [np.ones_like(alt), 1 - fv, 1 - fh, (1 - fv) * (1 - fh)]
    for k, dv in enumerate(DEV):
        TAB[az, k] = ((dni * cosi * unshaded[k]).sum() + (dhi * 0.5 * SKY[dv]).sum()) / 1000

G_REF = TAB[0, 0]
print('G_REF =', round(G_REF, 1), 'kWh/m2')

def gains(az, blk):
    # gain of each device for each plot, reduced by the share blocked by neighbours (Rad_block)
    i = np.round(np.nan_to_num(az)).astype(int) % 360
    return TAB[i] * (1 - np.nan_to_num(blk))[:, None]

G_REF = 239.5 kWh/m2


## 5. Facade decisions: shading, WWR and materials

For every existing facade (a direction with no wall stays empty):

**1. Shading** – the simplest device (none → horizontal → vertical → egg-crate) whose gain is ≤ G_REF. If none reaches it, the device with the lowest gain.

**2. WWR** – the glazing that keeps the window gain at the reference level: `WWR = min(0.30 × G_REF / gain_none, cap)`, cap = 0.30 within ±45° of north, otherwise 0.25. Classes: ≤ 0.10 small, ≤ 0.20 medium, else large.

**5. Materials**
- Wall, by summer radiation on the facade (`Rad_summer`): < 142 standard colour; < 203 light reflective colour; otherwise ventilated double skin on E/W and light colour with extra insulation on the other sides.
- Thermal glass, by gain_none / G_REF: ≤ 1 double clear; ≤ 2 double Low-E; otherwise triple Low-E.
- Acoustic glass, by the noise score (0 / 1 / 2): standard; double with a wider cavity; laminated sealed double.

In [5]:
out = pd.DataFrame({'ID': df.ID})
for d in D:
    has = df[f'Rad_summer_{d}'].notna().values
    G = gains(df[f'az_{d}'].values, df[f'Rad_block_{d}'].values)
    out[f'gain_none_{d}'] = np.where(has, G[:, 0].round(1), np.nan)

    # 1 shading
    ok = G <= G_REF + 1e-9
    first = np.where(ok.any(1), ok.argmax(1), G.argmin(1))
    out[f'dec_shading_{d}'] = np.where(has, np.array(DEV)[first], None)

    # 2 window-to-wall ratio
    az = df[f'az_{d}'].values
    north = np.abs((az + 180) % 360 - 180) <= 45
    cap = np.where(north, 0.30, 0.25)
    wwr = np.minimum(0.30 * G_REF / np.maximum(G[:, 0], 1e-6), cap)
    out[f'dec_wwr_{d}'] = np.where(has, wwr.round(3), np.nan)
    out[f'wwr_class_{d}'] = np.where(has, np.select([wwr <= 0.10, wwr <= 0.20], ['صغيرة', 'متوسطة'], 'كبيرة'), None)

    # 5 materials
    rs = df[f'Rad_summer_{d}'].values
    wall = np.select([rs < 142, rs < 203], ['لون عادي', 'لون فاتح عاكس'],
                     'غلاف مزدوج مهوّى' if d in ('E', 'W') else 'لون فاتح مع عزل أعلى')
    out[f'dec_wall_{d}'] = np.where(has, wall, None)
    ratio = G[:, 0] / G_REF
    out[f'dec_glass_thermal_{d}'] = np.where(has, np.select([ratio <= 1, ratio <= 2], ['مزدوج عادي', 'مزدوج Low-E'], 'ثلاثي Low-E'), None)
    nz = np.nan_to_num(df[f'Noise_{d}'].values).astype(int).clip(0, 2)
    out[f'dec_glass_acoustic_{d}'] = np.where(has, np.array(['عادي', 'مزدوج بفراغ أوسع', 'مزدوج مصفّح ومحكم الإغلاق'])[nz], None)

## 6. Plot decisions: main facade orientation and garden

Both decisions choose one side of the plot with the same procedure:

1. **Candidates**: wall sides (orientation) or yard sides (garden).
2. **Noise exclusion**: sides with a noise score ≥ 1 are removed, unless all candidates are noisy.
3. **Two criteria with rank-order weights 0.75 / 0.25** (Barron & Barrett, 1996), each min–max normalised within the plot (lowest value = best = 1).
   - Orientation: 0.75 × summer radiation on the wall + 0.25 × radiation blocked by neighbours.
   - Garden: 0.75 × summer radiation on the yard + 0.25 × blocking of the comfortable wind.
4. **Courtyard**: if the wall behind the best garden side has summer radiation ≥ 170, the garden becomes an inner courtyard.

In [6]:
W_A, W_B = 0.75, 0.25

def mm_best_low(v):
    # min-max within the plot: lowest value = 1, highest = 0; all equal -> 1
    lo, hi = np.nanmin(v, axis=1, keepdims=True), np.nanmax(v, axis=1, keepdims=True)
    r = hi - lo
    with np.errstate(all='ignore'):
        s = np.where(r > 0, 1 - (v - lo) / r, 1.0)
    return np.where(np.isnan(v), np.nan, s)

def side_decision(cand, crit_a, crit_b, wa=W_A, wb=W_B):
    noise = np.nan_to_num(df[[f'Noise_{d}' for d in D]].values)
    quiet = cand & (noise < 1)
    use = np.where(quiet.any(1, keepdims=True), quiet, cand)
    a, b = np.where(use, crit_a, np.nan), np.where(use, crit_b, np.nan)
    sa, sb = mm_best_low(a), mm_best_low(b)
    sb = np.where(use & np.isnan(sb), 0.5, sb)
    score = wa * sa + wb * sb
    ok = use.any(1)
    best = np.where(ok, np.nanargmax(np.where(use, score, -1), axis=1), -1)
    return best, score, use, ok

RS = df[[f'Rad_summer_{d}' for d in D]].values
RB = df[[f'Rad_block_{d}' for d in D]].values
RY = df[[f'Rad_yard_summer_{d}' for d in D]].values
WG = df[[f'Wind_block_good_{d}' for d in D]].values
AZ = df[[f'az_{d}' for d in D]].values
walls, yards = ~np.isnan(RS), ~np.isnan(RY)

# 4 main facade orientation
ob, _, _, ook = side_decision(walls, RS, RB)
out['dec_orient_side'] = np.where(ook, np.array(D)[np.clip(ob, 0, 7)], None)
out['dec_orient_deg'] = np.where(ook, AZ[np.arange(n), np.clip(ob, 0, 7)], np.nan)

# 3 garden
gb, _, _, gok = side_decision(yards, RY, WG)
court = gok & (RS[np.arange(n), np.clip(gb, 0, 7)] >= 170)
out['dec_garden'] = np.where(court, 'فناء داخلي', np.where(gok, np.array(D)[np.clip(gb, 0, 7)], None))

## 7. Stability check: swapping the two weights

The orientation and garden decisions are recomputed with the weights swapped (0.25 / 0.75). The share of plots whose decision does not change measures how sensitive these labels are to the order of the criteria.

In [7]:
ob2, _, _, _ = side_decision(walls, RS, RB, wa=W_B, wb=W_A)
gb2, _, _, _ = side_decision(yards, RY, WG, wa=W_B, wb=W_A)
court2 = gok & (RS[np.arange(n), np.clip(gb2, 0, 7)] >= 170)
garden2 = np.where(court2, 'فناء داخلي', np.where(gok, np.array(D)[np.clip(gb2, 0, 7)], None))
out['stable_orient'] = np.where(ook, (ob == ob2).astype(int), np.nan)
out['stable_garden'] = np.where(gok, (out.dec_garden.values == garden2).astype(int), np.nan)
print('orientation unchanged: %.1f%%' % (100 * np.nanmean(out.stable_orient)))
print('garden unchanged:      %.1f%%' % (100 * np.nanmean(out.stable_garden)))

orientation unchanged: 33.5%
garden unchanged:      55.5%


## 8. Train / test split by district

Neighbouring plots share the same buildings, streets and weather, so a random split would put near-identical plots in both sets. The data are split by **official Riyadh district** instead.

- Each plot is matched to the district polygon that contains it (a few plots at the city edge fall in districts of neighbouring towns); plots outside every polygon get the nearest district.
- **Test set**: 19 whole districts (about 20% of the plots), fixed below and used once for the final evaluation. They were chosen so the share is close to 20% and every class appears in the test set.
- **Training set**: the remaining districts, split into 5 folds with `GroupKFold` (each fold holds whole districts).

In [8]:
dist = gpd.read_file(DISTRICTS)
dist = dist[['district_id', 'name_ar', 'name_en', 'geometry']]   # all districts: a few edge plots fall in neighbouring cities
pts_g = gpd.GeoDataFrame(geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326)
dj = gpd.sjoin(pts_g, dist.to_crs(4326), how='left', predicate='within')
dj = dj[~dj.index.duplicated()]
outside = dj.district_id.isna()
if outside.any():
    nr = gpd.sjoin_nearest(pts_g[outside.values].to_crs(32638), dist.to_crs(32638), how='left')
    nr = nr[~nr.index.duplicated()]
    dj.loc[nr.index, ['district_id', 'name_ar', 'name_en']] = nr[['district_id', 'name_ar', 'name_en']].values
print('plots outside a boundary (nearest district used):', int(outside.sum()))

out['district'] = dj.name_ar.values
out['district_en'] = dj.name_en.values
out['district_id'] = dj.district_id.astype('int64').values

TEST_DISTRICTS = [10100003014, 10100003019, 10100003027, 10100003030, 10100003038, 10100003047,
                  10100003058, 10100003097, 10100003105, 10100003107, 10100003125, 10100003134,
                  10100003137, 10100003138, 10100003150, 10100003155, 10100003159, 10100003163,
                  10100003169]
is_test = out.district_id.isin(TEST_DISTRICTS).values
out['split'] = np.where(is_test, 'test', 'train')
out['fold'] = -1
tr = np.where(~is_test)[0]
for k, (_, va) in enumerate(GroupKFold(n_splits=5).split(tr, groups=out.district_id.values[tr])):
    out.loc[tr[va], 'fold'] = k
out['label_version'] = 'v1.2'
print(out.split.value_counts().to_dict(), '| districts:', out.district_id.nunique())
print(out[out.split == 'train'].groupby('fold').agg(plots=('ID', 'size'), districts=('district_id', 'nunique')))

plots outside a boundary (nearest district used): 154
{'train': 18244, 'test': 4695} | districts: 134
      plots  districts
fold                  
0      3649         22
1      3649         23
2      3650         23
3      3648         24
4      3648         23


## 9. Save the label files

In [9]:
DEC = ['dec_shading', 'dec_wwr', 'wwr_class', 'dec_wall', 'dec_glass_thermal', 'dec_glass_acoustic']
lab_cols = ['ID', 'label_version', 'dec_orient_side', 'dec_orient_deg', 'dec_garden'] + \
           [f'{p}_{d}' for p in DEC for d in D] + ['split', 'fold', 'district', 'district_en', 'district_id']
out[lab_cols].to_csv(f'{OUT_DIR}/phase2_labels_v1.2.csv', index=False, encoding='utf-8-sig')

trace = out[['ID'] + [f'gain_none_{d}' for d in D] + ['stable_orient', 'stable_garden']].copy()
trace['plot_axis'] = df.plot_axis
for d in D:
    trace[f'az_{d}'] = df[f'az_{d}']
trace.to_csv(f'{OUT_DIR}/phase2_trace_v1.2.csv', index=False, encoding='utf-8-sig')

rows = []
for d in D:
    m = out[f'dec_shading_{d}'].notna()
    t = out.loc[m, ['ID'] + [f'{p}_{d}' for p in DEC] + ['district', 'district_id', 'split', 'fold']].copy()
    t.columns = ['ID'] + DEC + ['district', 'district_id', 'split', 'fold']
    t.insert(1, 'facade_dir', d)
    t.insert(2, 'facade_az', df.loc[m, f'az_{d}'].values)
    rows.append(t)
F = pd.concat(rows).sort_values(['ID', 'facade_dir'])
F.to_csv(f'{OUT_DIR}/phase2_facade_labels_v1.2.csv', index=False, encoding='utf-8-sig')
print('plot file:', out[lab_cols].shape, '| facade file:', F.shape)

plot file: (22939, 58) | facade file: (92970, 13)


## 10. Summary and class distribution

In [10]:
def dist_of(s):
    vc = s.dropna().value_counts()
    return {str(k): [int(v), round(100 * v / vc.sum(), 1)] for k, v in vc.items()}

S = dict(G_REF=round(float(G_REF), 1), plots=n, facades=len(F),
         plot_file_columns=len(lab_cols), facade_file_columns=F.shape[1],
         no_orient=int((~ook).sum()), no_garden=int((~gok).sum()),
         orient=dist_of(out.dec_orient_side), garden=dist_of(out.dec_garden),
         shading=dist_of(F.dec_shading), wwr_class=dist_of(F.wwr_class), wall=dist_of(F.dec_wall),
         glass_thermal=dist_of(F.dec_glass_thermal), glass_acoustic=dist_of(F.dec_glass_acoustic),
         wwr_stats=dict(min=float(F.dec_wwr.min()), median=float(F.dec_wwr.median()), max=float(F.dec_wwr.max())),
         stable_orient_pct=round(100 * np.nanmean(out.stable_orient), 1),
         stable_garden_pct=round(100 * np.nanmean(out.stable_garden), 1),
         districts=int(out.district_id.nunique()), test_districts=len(TEST_DISTRICTS),
         split_plots=out.split.value_counts().to_dict(),
         facades_per_plot={int(k): int(v) for k, v in F.groupby('ID').size().value_counts().sort_index().items()})
json.dump(S, open(f'{OUT_DIR}/summary_v1.2.json', 'w'), ensure_ascii=False, indent=1)
print(json.dumps(S, ensure_ascii=False, indent=1))

{
 "G_REF": 239.5,
 "plots": 22939,
 "facades": 92970,
 "plot_file_columns": 58,
 "facade_file_columns": 13,
 "no_orient": 0,
 "no_garden": 194,
 "orient": {
  "NW": [
   6747,
   29.4
  ],
  "SE": [
   4351,
   19.0
  ],
  "SW": [
   3497,
   15.2
  ],
  "S": [
   3028,
   13.2
  ],
  "N": [
   2523,
   11.0
  ],
  "NE": [
   2114,
   9.2
  ],
  "W": [
   352,
   1.5
  ],
  "E": [
   327,
   1.4
  ]
 },
 "garden": {
  "فناء داخلي": [
   9110,
   40.1
  ],
  "SW": [
   2330,
   10.2
  ],
  "NW": [
   2286,
   10.1
  ],
  "SE": [
   2053,
   9.0
  ],
  "NE": [
   1713,
   7.5
  ],
  "E": [
   1501,
   6.6
  ],
  "S": [
   1438,
   6.3
  ],
  "W": [
   1327,
   5.8
  ],
  "N": [
   987,
   4.3
  ]
 },
 "shading": {
  "أفقي": [
   41504,
   44.6
  ],
  "شبكي": [
   35362,
   38.0
  ],
  "بدون": [
   15939,
   17.1
  ],
  "رأسي": [
   165,
   0.2
  ]
 },
 "wwr_class": {
  "متوسطة": [
   55043,
   59.2
  ],
  "كبيرة": [
   37927,
   40.8
  ]
 },
 "wall": {
  "لون عادي": [
   31017,
   33.4
